In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding

In [9]:
path = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
)

text = open(path, "rb").read().decode("utf-8")

In [10]:
charset = sorted(set(text))

char_to_int = {
    char: i 
    for i, char in enumerate(charset)
}

int_to_char = np.array(charset)

In [11]:
char_to_int

{'\n': 0,
 ' ': 1,
 '!': 2,
 '$': 3,
 '&': 4,
 "'": 5,
 ',': 6,
 '-': 7,
 '.': 8,
 '3': 9,
 ':': 10,
 ';': 11,
 '?': 12,
 'A': 13,
 'B': 14,
 'C': 15,
 'D': 16,
 'E': 17,
 'F': 18,
 'G': 19,
 'H': 20,
 'I': 21,
 'J': 22,
 'K': 23,
 'L': 24,
 'M': 25,
 'N': 26,
 'O': 27,
 'P': 28,
 'Q': 29,
 'R': 30,
 'S': 31,
 'T': 32,
 'U': 33,
 'V': 34,
 'W': 35,
 'X': 36,
 'Y': 37,
 'Z': 38,
 'a': 39,
 'b': 40,
 'c': 41,
 'd': 42,
 'e': 43,
 'f': 44,
 'g': 45,
 'h': 46,
 'i': 47,
 'j': 48,
 'k': 49,
 'l': 50,
 'm': 51,
 'n': 52,
 'o': 53,
 'p': 54,
 'q': 55,
 'r': 56,
 's': 57,
 't': 58,
 'u': 59,
 'v': 60,
 'w': 61,
 'x': 62,
 'y': 63,
 'z': 64}

In [12]:
int_to_char

array(['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?',
       'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
       'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
       'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
       'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z'],
      dtype='<U1')

In [13]:
# convert the complete text

text_as_int = np.array(
    [char_to_int[c] for c in text],
    dtype=np.int32
)

In [14]:
print("Original Text: ", text[:50])
print("Text as Integers: ", text_as_int[:50])

Original Text:  First Citizen:
Before we proceed any further, hear
Text as Integers:  [18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56 43  1 61 43
  1 54 56 53 41 43 43 42  1 39 52 63  1 44 59 56 58 46 43 56  6  1 46 43
 39 56]


In [15]:
sequence_length = 100
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(
    sequence_length + 1,
    drop_remainder=True
)

In [16]:
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]

    return input_text, target_text

dataset = sequences.map(split_input_target)

In [20]:
# sequence

for input_example, target_example in dataset.take(1):

    print("INPUT:")
    print("".join(int_to_char[input_example.numpy()]))

    print("\nTARGET:")
    print("".join(int_to_char[target_example.numpy()]))

INPUT:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

TARGET:
irst Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You 


In [21]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

In [23]:
charset_size = len(charset)

embedding_dim = 256
lstm_units = 512

model = tf.keras.models.Sequential([
    
    tf.keras.layers.Embedding(
        input_dim=charset_size,
        output_dim=embedding_dim
    ),

    tf.keras.layers.LSTM(
        lstm_units,
        return_sequences=True
    ),

    tf.keras.layers.Dense(charset_size)
])

model.build(input_shape=(None, sequence_length))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 100, 256)            │          16,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 100, 512)            │       1,574,912 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 100, 65)             │          33,345 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,624,897 (6.20 MB)

 Trainable params: 1,624,897 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

model.compile(
    optimizer="adam",
    loss=loss_fn
)

In [26]:
EPOCHS = 10

history = model.fit(
    dataset,
    epochs=EPOCHS
)

Epoch 1/10


C:\Users\regmi\Documents\UCMO\Neural Networks and Deep Learning\env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


172/172 ━━━━━━━━━━━━━━━━━━━━ 90s 508ms/step - loss: 2.5688
Epoch 2/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 104s 600ms/step - loss: 1.9406
Epoch 3/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 107s 615ms/step - loss: 1.7281
Epoch 4/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 105s 604ms/step - loss: 1.6082
Epoch 5/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 103s 591ms/step - loss: 1.5298
Epoch 6/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 93s 534ms/step - loss: 1.4746
Epoch 7/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 95s 547ms/step - loss: 1.4328
Epoch 8/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 101s 582ms/step - loss: 1.3999
Epoch 9/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 101s 576ms/step - loss: 1.3715
Epoch 10/10
172/172 ━━━━━━━━━━━━━━━━━━━━ 118s 678ms/step - loss: 1.3480


In [16]:
def generate_text(model, start_string, num_generate=500, temperature=1.0):

    input_ids = [
        char_to_int[c]
        for c in start_string
        if c in char_to_int
    ]

    input_ids = tf.expand_dims(input_ids, 0)

    generated_text = list(start_string)

    for _ in range(num_generate):

        predictions = model(input_ids, training=False)

        predictions = predictions[:, -1, :]

        predictions = predictions / temperature

        predicted_id = tf.random.categorical(
            predictions,
            num_samples=1
        )[0, 0].numpy()

        predicted_char = int_to_char[predicted_id]

        generated_text.append(predicted_char)

        predicted_id_tensor = tf.expand_dims(
            [predicted_id],
            0
        )

        input_ids = tf.concat(
            [input_ids, predicted_id_tensor],
            axis=1
        )

        input_ids = input_ids[:, -sequence_length:]

    return "".join(generated_text)

In [ ]:
generated = generate_text(
    model,
    start_string="ROMEO:",
    num_generate=1000,
    temperature=1.0
)

print(generated)